# 02 - Prétraitement des volumes BraTS

Ce notebook charge les volumes NIfTI, normalise les intensités et crée des séparations patient-level reproductibles. Les données originales ne sont jamais modifiées.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "data/raw/brats2020-training-data"
OUTPUT_ROOT = REPO_ROOT / "results/preprocessed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
from brain_mri_fem.preprocessing import load_nifti, normalize_volume, patient_splits, save_splits
print(f"Dataset: {DATA_ROOT}")
print(f"NIfTI disponibles: {len(list(DATA_ROOT.rglob('*.nii*')))}")

In [ ]:
def patient_id(path: Path) -> str:
    name = path.name.lower().replace(".nii.gz", "").replace(".nii", "")
    for suffix in ("_flair", "_t1ce", "_t1", "_t2", "_seg"):
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return name

flair_files = sorted(DATA_ROOT.rglob("*_flair.nii*"))
if not flair_files:
    print("Aucun volume NIfTI trouvé. Exécutez le téléchargement BraTS puis relancez cette cellule.")
else:
    ids = [patient_id(path) for path in flair_files]
    splits = patient_splits(ids, seed=42)
    save_splits(splits, OUTPUT_ROOT / "splits.json")
    print({name: len(values) for name, values in splits.items()})

In [ ]:
if flair_files:
    image, affine, spacing = load_nifti(flair_files[0])
    normalized = normalize_volume(image)
    print("Forme:", image.shape, "Spacing:", spacing)
    print("Avant:", float(image[image != 0].mean()), float(image[image != 0].std()))
    print("Après:", float(normalized[normalized != 0].mean()), float(normalized[normalized != 0].std()))
    index = image.shape[2] // 2
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image[:, :, index].T, cmap="gray", origin="lower")
    axes[0].set_title("Original")
    axes[1].imshow(normalized[:, :, index].T, cmap="gray", origin="lower")
    axes[1].set_title("Z-score non-zero")
    for axis in axes:
        axis.axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / "normalization_example.png", dpi=150)
    plt.show()